# 6 — A 60- and 90-day adjoint: chunking, freezing, and knowing when to believe it

Notebook 5 differentiated a **five-day** window of the global ocean. This one goes to **sixty and
ninety days**, which is a different problem in three ways rather than the same problem for longer:

1. **The reverse pass no longer fits on the card**, and the usual fix does not help.
2. **The gradient stops being usable long before it stops being finite** — you have to freeze
   three specific things, and most plausible choices make it worse.
3. **A finite, reproducible gradient can still be worthless**, and the only way to know is to
   read a diagnostic the reverse sweep produces on the way past.

Each section gives the measurement behind the advice, because most of these are not obvious and
several are counter-intuitive.

> **Notebook 5 first** if you have not read it — it covers the one-day building block, why a
> `stop_gradient` on a diffusivity is safe, and the finite-difference check. This notebook assumes
> that material and concentrates on what changes with length.

## What you need

- **The CORE2 data from Zenodo** ([doi:10.5281/zenodo.21324319](https://doi.org/10.5281/zenodo.21324319)),
  fetched exactly as in notebook 5:

  ```bash
  python scripts/fetch_data.py --dest ~/fesom-data
  eval "$(python scripts/fetch_data.py --dest ~/fesom-data --print-env)"
  ```

- **A GPU with 80 GB.** The device peak is one chunk — about 43 GB — regardless of how long the
  window is, so this is a floor and not something that grows on you.

- **Host RAM**, which *is* what grows: one saved chunk boundary is a CORE2 `State`, **1.93 GB**.
  Section 5 is entirely about that.

- **Time.** Set `N_DAYS` below. The default is deliberately small so the notebook runs; the long
  configurations are the point of the notebook but belong in a batch job, and Section 8 gives the
  measured costs.

In [ ]:
import os, time
import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt

import fesom_jax                      # importing turns on float64
from fesom_jax import ale, adjoint_stab, ssh, surface_forcing
from fesom_jax.adjoint import chunked_forward, chunked_grad, scan_steps
from fesom_jax.adjoint_stab import AdjointConfig
from fesom_jax.calibrate import build_params
from fesom_jax.gm import GMConfig
from fesom_jax.ice import IceConfig
from fesom_jax.mesh import load_mesh
from fesom_jax.params import Params
from fesom_jax.phc_ic import phc_initial_state
from fesom_jax.step import step
from fesom_jax.tke import TkeConfig

# ============================================================================
# THE THREE KNOBS. Everything else in this notebook is derived from them, so
# changing a window length here is one edit and not a hunt.
# ============================================================================
N_DAYS = int(os.environ.get("N_DAYS", 2))            # window length
CHUNK_DAYS = float(os.environ.get("CHUNK_DAYS", 1))  # level 2: chunk = this many days
BOUNDARY_STRIDE = int(os.environ.get("STRIDE", 1))   # level 3: keep every Nth boundary
# ============================================================================

DT = 1800.0                                   # seconds
STEPS_PER_DAY = 48                            # 86400 / DT
N_STEPS = N_DAYS * STEPS_PER_DAY
CHUNK_STEPS = max(1, int(round(CHUNK_DAYS * STEPS_PER_DAY)))

# ⚠️ Derived, NOT `N_CHUNKS = N_DAYS`. Writing that assumes one chunk per day, and then changing
# CHUNK_DAYS silently integrates a different window than the one you asked for — the run still
# succeeds and the gradient is of the wrong thing.
if N_STEPS % CHUNK_STEPS:
    raise SystemExit(f"{N_STEPS} steps is not a whole number of {CHUNK_STEPS}-step chunks; "
                     f"pick a CHUNK_DAYS that divides N_DAYS")
N_CHUNKS = N_STEPS // CHUNK_STEPS

MESH_DIR = os.environ["FESOM_MESH_DIR"]       # from fetch_data.py --print-env
IC_DIR = os.environ["FESOM_IC_DIR"]

STATE_GB = 1.93                               # a CORE2 State; confirmed against the mesh below

def host_states(n_chunks, stride):
    '''Peak host residency, in States, for a given stride.

    ⚠️ NOT just the stored boundaries. While a span is rebuilt the library holds that span's
    states ALONGSIDE the kept ones, so the peak is `kept + stride`, which is the `n/stride +
    stride` the docs quote and is minimised near sqrt(n). Counting only the kept boundaries
    understates a year at stride 19 as 40 GB when the peak is ~77 GB — and a job sized from
    the smaller number dies partway through the reverse sweep.'''
    kept = len(range(0, n_chunks, stride)) + 1
    return kept + (stride if stride > 1 else 0)

peak = host_states(N_CHUNKS, BOUNDARY_STRIDE)
best_stride = max(1, round(N_CHUNKS ** 0.5))
best_peak = host_states(N_CHUNKS, best_stride)

print("device ", jax.devices()[0])
print(f"window  {N_DAYS} d = {N_STEPS} steps = {N_CHUNKS} chunks x {CHUNK_STEPS} steps "
      f"({CHUNK_DAYS} d)")
print(f"host    stride {BOUNDARY_STRIDE}: peak ~{peak} States ~ {peak * STATE_GB:.0f} GB")
if BOUNDARY_STRIDE != best_stride:
    print(f"        stride {best_stride} (= round(sqrt({N_CHUNKS}))) would peak at ~{best_peak} "
          f"~ {best_peak * STATE_GB:.0f} GB, for one extra forward pass")
print(f"device  ~43 GB, one chunk, independent of the window length")

## 1. Why `jax.grad` over the whole window does not work

The obvious thing is to write the window as one function and call `jax.grad` on it. That fails,
and it fails for a reason worth understanding, because the standard remedy also fails.

Reverse-mode AD has to keep enough of the forward trajectory to run the backward. The standard
trade is **two-level rematerialization**: split `N` steps into `S` segments, store one carry per
segment and one per step inside the segment, recompute the rest. That stores `S + N/S` carries,
minimised at `S = √N`.

Now put the numbers in. A CORE2 `State` is **1.93 GB**. A five-day window is `N = 240` steps, so
even at the optimum:

```
2 · √240 · 1.93 GB  ≈  60 GB   of carries
                    +  ~18 GB  of reverse working set
                    =  ~78 GB  against ~78 GB usable on an 80 GB A100
```

Five days is already at the edge — and **no value of `S` fixes it**. Larger `S` means more outer
carries, smaller `S` means more inner ones; `√N` is the floor, not a starting guess. Measured:
`S = 31` died at 69.24 GiB.

That is why `fesom_jax.adjoint.scan_steps` has **no `segments` parameter**. It derives `√N` and
does not offer the knob, because the knob reads exactly like the thing you would reach for when a
window will not fit, and reaching for it makes things worse.

## 2. What does work: chunk the window, park the boundaries on the host

Run the window forward in chunks of `CHUNK_STEPS`, keep each chunk-boundary `State` **on the
host**, then walk the chunks backwards calling a jitted `jax.vjp` on one chunk at a time, carrying
the state cotangent and summing the parameter cotangent.

The device then holds one chunk, whatever the window length. Measured on an 80 GB A100:

| window | 1 day | 30 days | 60 days |
|---|---|---|---|
| device peak | 43.1 GB | 43.2 GB | 43.1 GB |

Flat. The cost has moved to host memory, which Section 5 deals with.

**This is exact.** It is checkpointing, not an approximation — the chunked gradient equals a
monolithic `jax.grad` over a window where both fit, and it is independent of the chunk length.
Both are asserted in the library's own tests.

## 3. The three freezes, and why most alternatives are worse

Left alone, the reverse pass over a long window grows by five to seven orders of magnitude and the
gradient is meaningless. Three things have to be frozen — meaning *use the value this term
computes, but do not trace how it depended on the ocean state*. The forward trajectory is
untouched by all three; `stop_gradient` is the identity going forwards.

| # | what is frozen | how |
|---|---|---|
| 1 | the **mixing coefficients** (TKE closure output) | `AdjointConfig(freeze_mixing=True)` |
| 2 | the **GM neutral-slope factor** | `AdjointConfig(freeze_gm_slope=True)` |
| 3 | the **mEVP ice momentum sweeps** | `IceConfig(adjoint_mode="rheology")` |

Two of those are ocean and live in one object; the third is sea ice and has its own switch, which
predates `AdjointConfig`. The forward still runs all 120 mEVP relaxation sweeps — only the backward
unrolls zero of them.

### Why not something else?

This is the part that matters, and it is measured rather than argued. On a 60-day window with
`freeze_mixing` already on, against a baseline cotangent growth of ×4.363e+07:

| flag | effect at 60 days |
|---|---|
| `freeze_gm_slope` | **×1.591 — the whole cure** |
| `freeze_stratification` | ×1.590 — the same, it subsumes the above |
| `freeze_advection` | ×3.463e+07 — a real 21 %, still blown |
| `freeze_vert_visc` | ×3.541e+07 — 19 %, still blown |
| `freeze_k33` | ×8.092e+07 — **1.85× worse** |
| `freeze_convection` | ×9.254e+07 — **2.12× worse** |
| `freeze_tke_state` | ×1.275e+08 — **2.92× worse** |
| `freeze_pgf` | ×1.145e+09 — **26× worse** |

Of the eight flags, only `freeze_gm_slope` and the wider `freeze_stratification` that contains it
produce a usable gradient. **Four of the eight make things worse than not cutting at all** — they
sit inside stabilising loops, and removing the feedback removes the stabilisation.

⚠️ There is an older two-day table in the same docstring that records `freeze_gm_slope` as
*"1.4298, no effect"*. That verdict is an artefact: on a two-day window the unfrozen scatter is
1.21–1.55, so any flag landing inside that band was recorded as "no effect" when the truth is that
the experiment could not tell. **Read it only as "this did not rescue an otherwise unusable two-day
gradient", never as evidence about a configuration that already works.**

In [ ]:
# The three freezes. Nothing here changes the forward trajectory.
adj = AdjointConfig(freeze_mixing=True, freeze_gm_slope=True)   # ocean: 1 and 2
ice = IceConfig(whichEVP=1, adjoint_mode="rheology")            # ice: 3

gm, tke, ale_cfg = GMConfig(), TkeConfig(), ale.AleConfig()

print("ocean freezes :", adj.describe())
print("ice           :", ice.adjoint_mode,
      "- thermodynamics, advection and the ocean coupling stay differentiated;")
print("                only the mEVP momentum solve is dark in the backward")

## 3b. Three levels of checkpointing, and which two you tune

"Chunking" is not one thing you switch on. There are **three nested levels**, and it is worth
knowing which is which, because they trade against different resources.

| level | what it does | knob | who sets it |
|---|---|---|---|
| **1. within a chunk** | √N rematerialization inside the `lax.scan` — store ~2√K carries instead of K | none | the library, always on |
| **2. the chunk** | how many steps run before a boundary is parked on the host | `CHUNK_DAYS` | you |
| **3. the boundary** | keep every Nth boundary, recompute the spans between | `BOUNDARY_STRIDE` | you |

**Level 1 is not a knob and should not be.** It is `scan_steps(rematerialize=True)`, and √N is
already the memory optimum — that is Section 1. The only real choice is on/off, and off means the
chunk's whole trajectory is kept, which is what does not fit. `rematerialize=False` exists for
forward-only and forward-mode runs, where there is no reverse pass to feed and the checkpointing
is pure overhead.

**Level 2 sets the device peak.** One chunk must fit on the card. Bigger chunks mean fewer host
boundaries and a higher device peak; smaller chunks the reverse. One day at 43 GB is a comfortable
setting for an 80 GB A100.

**Level 3 sets the host cost**, and it is the one that decides whether a long window is possible
at all. It costs exactly one extra forward pass over the window and changes no number.

### Turning the dials

Set the three at the top and re-run. Some working combinations:

| you want | `N_DAYS` | `CHUNK_DAYS` | `STRIDE` | peak host | device | rough wall |
|---|---|---|---|---|---|---|
| see it work | 2 | 1 | 1 | 6 GB | 43 GB | ~10 min |
| a real window | 30 | 1 | 1 | 60 GB | 43 GB | ~1 h |
| the same, lean | 30 | 1 | 5 | 23 GB | 43 GB | ~1 h + one forward |
| **60 days** | 60 | 1 | 8 | 33 GB | 43 GB | ~2 h |
| **90 days** | 90 | 1 | 9 | 39 GB | 43 GB | ~3 h |
| a year | 366 | 1 | 19 | 77 GB | 43 GB | ~10 h, needs resume |
| smaller card | 60 | 0.5 | 11 | 44 GB | **~22 GB** | slower |

These are what the config cell prints for those settings — it computes them, so you can see the
cost before committing a job rather than after.

⚠️ The last row trades the *other* way and it is worth seeing why. Halving the chunk halves the
device peak, which is what lets a 60-day window onto a 40 GB card — but it doubles the number of
chunks, so the host cost goes **up**, 33 GB to 44 GB. Level 2 and level 3 pull in opposite
directions; there is no setting that is cheap in both.

They are also environment variables, so a batch script needs no edit at all:

```bash
N_DAYS=60 STRIDE=8 python nb06_long_window.py
```

⚠️ `N_DAYS × 48` must be a whole number of `CHUNK_DAYS × 48` chunks — the config cell refuses
rather than quietly integrating a different window.

## 4. Build the model

Identical to notebook 5. The only long-window-specific choice is `CHUNK_STEPS`.

**Choosing the chunk length.** It sets the device peak (one chunk must fit) and the host cost (one
boundary per chunk). One day at `dt = 1800 s` is 48 steps, which measures at ~43 GB on an A100 and
1.93 GB of host per chunk. Shorter chunks lower the device peak and raise the host cost; longer
chunks do the reverse. It does **not** change the answer — the gradient is independent of it.

In [ ]:
mesh = load_mesh(MESH_DIR)
state0 = phc_initial_state(mesh, IC_DIR)          # observed winter T and S
op = ssh.build_ssh_operator(mesh, dt=DT)
cf = surface_forcing.build_surface_forcing(mesh, 1958, sst_ic=np.asarray(state0.T[:, 0]))
dates = surface_forcing.dates_for_steps(1958, DT, N_DAYS * STEPS_PER_DAY)

kgm0 = float(Params.defaults().k_gm)
wet = jnp.asarray(mesh.node_layer_mask[:, 0])

state_gb = sum(a.nbytes for a in jax.tree.leaves(jax.device_get(state0))) / 1e9
print(f"{mesh.nod2D:,} surface points, {mesh.nl} levels")
print(f"one State = {state_gb:.2f} GB on the host  ->  "
      f"{(N_CHUNKS + 1) * state_gb:.1f} GB for {N_CHUNKS + 1} boundaries at stride 1")

## 5. The three things you supply

`chunked_grad` owns the loop, the buffer lifetimes and the boundary bookkeeping. It cannot know
your model, so you pass three functions.

**`chunk_fn(state, xs, params) -> state`** — advance the state by one chunk. Everything
model-specific closes over here: mesh, SSH operator, configuration, which terms are frozen, how
your parameter becomes `Params`.

**`xs_fn(c) -> xs`** — the forcing for chunk `c`, **built on demand**. This is not a stylistic
preference:

> ⚠️ Building every chunk's forcing up front and holding it on the device costs about **0.5 GB per
> simulated day** of pure idle residency, and it is the entire reason the device peak used to grow
> with window length — 43.1 GB at one day rising to 56.9 GB at thirty, extrapolating to an OOM
> around ninety days. Built per chunk and freed, the peak is flat, with bit-identical results:
> only *when the arrays exist* changed.

> ⚠️ **`chunked_grad` frees what `xs_fn` returns, and calls it more than once per chunk** (the
> reverse walk re-asks; `boundary_stride > 1` re-asks again while rebuilding a span). Build fresh
> each call. Returning a cached array — or a slice of one, since in JAX `a[0:len(a)]` **is** `a` —
> hands the library a buffer it will delete. You get an error naming this contract, not silent
> corruption, and `free_xs=False` opts out.

**`loss_fn(final_state) -> scalar`** — differentiated once, to seed the reverse sweep.

In [ ]:
def chunk_fn(state, xs, m):
    """One chunk. `m` scales the eddy diffusivity; it is what we differentiate."""
    p = build_params({"k_gm": kgm0 * m, "redi_kmax": kgm0 * m})

    def one_step(s, sf):
        return step(s, mesh, op, None, p, dt=DT, is_first_step=False,
                    step_forcing=sf, forcing_static=cf.static,
                    ice_cfg=ice, gm_cfg=gm, tke_cfg=tke, ale_cfg=ale_cfg,
                    adjoint_cfg=adj, remat_blocks=True)

    # rematerialize=True: this chunk WILL be differentiated in reverse mode, so the
    # sqrt(N)-within-chunk scheme is what we want. scan_steps derives sqrt(48) itself.
    return scan_steps(one_step, state, xs, rematerialize=True)


def xs_fn(c):
    """Chunk c's forcing, built fresh. See the warning above."""
    return cf.stack(dates[c * CHUNK_STEPS:(c + 1) * CHUNK_STEPS])


def loss_fn(s):
    """Mean sea-surface temperature over wet surface points."""
    return jnp.sum(jnp.where(wet, s.T[:, 0], 0.0)) / jnp.sum(wet)

## 6. Run it

`on_forward` and `on_chunk` are progress hooks. On a long window the reverse sweep runs for hours,
and without them the first output of the whole call is its return value.

In [ ]:
m0 = jnp.asarray(1.0, jnp.float64)

def on_forward(loss, host_gb, secs):
    print(f"[fwd] loss={loss:.6e}  {secs:.0f}s  boundaries on host: {host_gb:.1f} GB", flush=True)

def on_chunk(c, cot_norm, g):
    print(f"  [bwd] chunk {c:3d}  t=+{c * CHUNK_STEPS * DT / 86400:5.2f} d  "
          f"|cot|={cot_norm:.4e}  dL/dm so far={float(g):+.6e}", flush=True)

t0 = time.time()
res = chunked_grad(chunk_fn, m0,
                   init_state=state0, n_chunks=N_CHUNKS,
                   xs_fn=xs_fn, loss_fn=loss_fn, chunk_steps=CHUNK_STEPS,
                   boundary_stride=BOUNDARY_STRIDE,
                   on_forward=on_forward, on_chunk=on_chunk)

print(f"\nloss      {res.loss:.9e}")
print(f"dL/dm     {float(res.grad):+.9e}")
print(f"host      {res.host_gb:.1f} GB   forward {res.forward_seconds:.0f}s   "
      f"reverse {res.reverse_seconds:.0f}s   total {time.time() - t0:.0f}s")

## 7. `boundary_stride` — the knob that makes a year possible

Keeping one boundary per chunk costs `(n_chunks + 1) × 1.93 GB` of host memory. That is fine at 60
days (118 GB, measured) and impossible at a year:

```
366 days × 1.93 GB = 706 GB     against a 515 GB node
```

`boundary_stride = N` keeps every Nth boundary and **recomputes the spans between** from the
nearest kept one. The host cost becomes `n/stride + stride` states, minimised near `√n_chunks`:

| window | stride 1 | best stride | host at best |
|---|---|---|---|
| 60 days | 118 GB | 8 | ~29 GB |
| 366 days | 706 GB | 19 | **~74 GB** |

The price is exactly **one extra forward pass** over the window, because each chunk is recomputed
at most once across the whole sweep.

**The gradient does not change** — the recomputed states are the same states. Validated at strides
1, 2 and 3: `dL/dm` agrees to **3e-5**, which is tighter than the model's own forward
reproducibility of 6e-4.

In [ ]:
# What the arithmetic says for this window, and the check that it costs nothing but time.
for stride in (1, 2, 4):
    print(f"  stride {stride}: peak ~{host_states(N_CHUNKS, stride)} States, "
          f"{host_states(N_CHUNKS, stride) * state_gb:.1f} GB host")

# Cheap only because N_DAYS is small here — at 60 days this is a second full run.
if N_DAYS <= 4:
    alt = chunked_grad(chunk_fn, m0, init_state=state0, n_chunks=N_CHUNKS,
                       xs_fn=xs_fn, loss_fn=loss_fn, chunk_steps=CHUNK_STEPS,
                       boundary_stride=2)
    rel = abs(float(alt.grad) - float(res.grad)) / abs(float(res.grad))
    print(f"\nstride 1 : {float(res.grad):+.12e}   host {res.host_gb:.2f} GB")
    print(f"stride 2 : {float(alt.grad):+.12e}   host {alt.host_gb:.2f} GB")
    print(f"relative difference {rel:.2e}   (bar: 3e-5)")

## 8. Is the gradient usable? Read the trace

**This is the section to not skip.** A long-window gradient can be finite, reproducible to six
significant figures, and still worthless — and it is the **seed state** that decides. With no
network and no other change, the same probe amplified the reverse pass at 1.0008 per step from one
seed and 1.0539 per step from another. Over a 240-step window that is a factor of 1.2 against a
factor of 3×10⁵.

So every window gets screened before its gradient is used. `chunked_grad` returns the
chunk-boundary cotangent norms as `res.trace` — the reverse sweep produces them anyway — and
`adjoint_stab.amplification` turns them into per-step rates.

**Two statistics, because they fail differently.** A raised **median** is smooth exponential
growth: chaos. A raised **log-spread** at a normal median is a switch firing in one chunk. A
median-only screen misses the second case entirely — and that is not hypothetical: two flag
settings record a median of exactly 1.0000/step against a 1.010 bar and are caught only by the
spread, while returning `dL/dm = -2.007e+05` against a true finite difference of `-9.80e-01`.

In [ ]:
amp = adjoint_stab.amplification(res.trace, res.chunk_steps)

# ⚠️ The log-spread is a standard deviation ACROSS CHUNKS. On a short window there are not
# enough of them for it to mean anything, and it will happily return a REJECT that is an
# artefact of having two samples. The screen is built for the 60-90 chunk windows this
# notebook is about; below ~10 it is reporting noise.
if N_CHUNKS < 10:
    print(f"!! {N_CHUNKS} chunks: the log-spread below is computed from {N_CHUNKS} samples and")
    print("!! carries no information. Read the median only, and do not trust the verdict.\n")

print(f"median      {amp.median:.6f} /step   (bar {adjoint_stab.AMP_CLEAN_MAX:.3f})")
print(f"log-spread  {amp.log_spread:.4f}        (bar {adjoint_stab.AMP_SPREAD_MAX:.3f})")
print(f"worst chunk {amp.worst_chunk}  at {amp.max:.6f} /step")
print(f"\nverdict: {'CLEAN' if amp.clean else 'REJECT - ' + amp.reason}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].semilogy(res.trace, marker="o", ms=3)
ax[0].set_xlabel("reverse sweep position (0 = loss seed)")
ax[0].set_ylabel("|cotangent|")
ax[0].set_title("cotangent norm along the reverse pass")
ax[1].axhline(1.0, color="0.7", lw=1)
ax[1].axhline(adjoint_stab.AMP_CLEAN_MAX, color="crimson", lw=1, ls="--", label="median bar")
ax[1].plot(amp.per_step, marker="o", ms=3)
ax[1].set_xlabel("chunk (reverse order)")
ax[1].set_ylabel("growth per step")
ax[1].set_title("per-step amplification")
ax[1].legend()
fig.tight_layout()

## 9. Check it against finite differences

The screen says the gradient is not blowing up. It does not say the gradient is *right*. For that,
perturb the parameter and re-run the forward.

`chunked_forward` is the forward half on its own — it parks no boundaries and writes to no
checkpoint, which is what these evaluations need. Both jits are cached on `chunk_fn`, so the
perturbed runs reuse the executable the gradient pass already compiled.

⚠️ Two full forwards per step size. At 60 days that is roughly a third of the total cost, so use
one `h` when sweeping configurations and widen it only for the one you end up trusting.

In [ ]:
def loss_at(m):
    fin = chunked_forward(chunk_fn, m, init_state=state0,
                          n_chunks=N_CHUNKS, xs_fn=xs_fn)
    return float(loss_fn(fin))

# FIRST: what can this comparison even resolve? The same forward, twice, unperturbed. On a GPU
# the model is not bit-reproducible (atomic scatter-adds on an unstructured mesh), so this is the
# floor below which an FD difference means nothing. Measure it rather than assume it — a check
# whose resolving power you have not established can only mislead you.
a, b = loss_at(m0), loss_at(m0)
noise = abs(b - a)
print(f"forward noise      {noise:.3e}  (same m, twice)")

h = 1e-2
lp, lm = loss_at(m0 * (1 + h)), loss_at(m0 * (1 - h))
signal = abs(lp - lm)
fd = (lp - lm) / (2 * h)
ad = float(res.grad)

print(f"FD signal          {signal:.3e}  (|loss(m+h) - loss(m-h)|)")
print(f"signal / noise     {signal / max(noise, 1e-300):.1f}")
print()
print(f"adjoint            {ad:+.6e}")
print(f"finite difference  {fd:+.6e}   (h = {h})")
print(f"relative           {abs(ad - fd) / abs(fd):.2e}")
print(f"signs agree        {np.sign(ad) == np.sign(fd)}")

if signal < 20 * noise:
    print("\n!! The FD signal is not clear of the forward's own noise, so the relative")
    print("!! difference above is measuring nondeterminism, not the adjoint. Lengthen the")
    print("!! window or raise h; do not read a verdict off this.")

### What this actually printed at `N_DAYS = 2`

Run on an A100 while writing this notebook, so you can tell a working setup from a broken one:

```
one State = 1.93 GB on the host  ->  5.8 GB for 3 boundaries at stride 1
[fwd] loss=1.078282e+01  237s  boundaries on host: 5.8 GB
dL/dm     +1.378307156e-03        forward 237s   reverse 293s
stride 1 : +1.378307156264e-03    host 5.80 GB
stride 2 : +1.378307189557e-03    host 3.86 GB
relative difference 2.42e-08      (bar: 3e-5)
median 0.993332 /step   log-spread 0.0227   verdict: REJECT
adjoint +1.378307e-03   finite difference +6.783168e-04   relative 1.03
```

**The stride agreement is the part that means something** — 2.4e-08 against a 3e-5 bar says the
recompute path is exact, and that claim does not care how long the window is.

**The REJECT is an artefact** — the log-spread is a standard deviation over chunks and there are
two of them. The cell says so itself now.

**The failed FD check is not an artefact, and finding that out is the point of measuring the noise
floor.** My first guess was that a two-day signal is lost in GPU nondeterminism. The cell disagrees:

```
forward noise   8.363e-12   (the same forward, twice, same m)
FD signal       1.357e-05
signal / noise  1.6e+06
```

The forward is essentially deterministic on this window, and the FD has six orders of magnitude of
headroom. So the factor of 2.03 between the adjoint and the finite difference is **real**, and the
explanation is the next section: it is what the freezes cost.

Two things worth carrying from that. **A diagnostic that has not been shown to resolve anything is
not a diagnostic** — which is why both cells now measure their own resolving power first. And
**"it is probably noise" is a hypothesis, not an answer**; it took one extra pair of forward runs
to find out it was wrong.

## The numbers that do mean something

With the three freezes on, a 60-day window agrees with the true finite difference to about **4 %**.
That is the number to have in mind: the freezes do not give you the exact gradient of the unmodified
model, they give you a usable descent direction for one that would otherwise be noise. Whether 4 %
is good enough is a question about your optimiser, not about the adjoint.

## 10. Windows longer than the queue

At a year the reverse sweep is around 10 hours against a 12-hour GPU limit, and two years fit no
slot at all. But the entire carried state of the reverse loop is one cotangent `State`, the running
parameter cotangent and an index — so it can stop and continue.

`chunked_grad` takes a `checkpoint=` object implementing the `ReverseCheckpoint` protocol:

```python
have_boundaries: bool                         loss: float
forward_resume(like)     -> (state | None, first_chunk)
put_boundary(c, state)   -> bool              # True => stop the leg here
load_boundaries(like)    -> dict
finish_forward(loss)     -> None
reverse_start(seed_cot, n_chunks, like) -> (cot, grad_so_far, start_chunk, trace)
tick(next_chunk, cot, grad, trace)      -> bool   # True => stop the leg here
```

The library **calls** this and does not implement it: a real one knows about filesystem layout, an
exit code your resubmit script recognises, and a wall-clock budget, none of which belong in a
library. When it asks to stop, `chunked_grad` raises `LegIncomplete` — raised rather than returned,
so a partial sweep can never be mistaken for a gradient.

⚠️ If you catch that exception, catch it **before** any general `except Exception`, and exit with
the code your resubmit script expects. A leg that did exactly what it was told, logged as a failure
and exited 0, stops the chain and abandons a good checkpoint.

A stopped-and-resumed sweep is asserted bit-identical to a single pass in the library's tests.

## 11. What it costs

Measured on one A100 80 GB, CORE2, `dt = 1800 s`, one-day chunks:

| window | device peak | host, stride 1 | host at √n | wall |
|---|---|---|---|---|
| 1 day | 43.1 GB | 3.9 GB | — | minutes |
| 30 days | 43.2 GB | 60 GB | 23 GB (stride 5) | ~1 h |
| 60 days | 43.1 GB | 118 GB | 33 GB (stride 8) | ~2 h |
| 90 days | 43.1 GB | 176 GB | 39 GB (stride 9) | ~3 h |
| 1 year | 43.1 GB | 708 GB — will not fit | 77 GB (stride 19) | ~10 h, needs resume |

Wall times include the perturbed forwards for one finite-difference step size. Add roughly one
forward pass over the window for any stride above 1.

**Clean gradients are proven out to 60 days.** Beyond that the machinery works and the screen still
applies, but the claim about usefulness has not been made.

## What to take away

- `jax.grad` over a long window does not fit, and **`segments` is not the fix** — `√N` is already
  the floor and both directions cost more.
- **Chunk it.** Device peak becomes one chunk and stops depending on window length.
- Build per-chunk forcing **on demand**; a list of chunks is what makes the peak grow again.
- **Three freezes**: `freeze_mixing`, `freeze_gm_slope`, `adjoint_mode="rheology"`. Four of the
  eight ocean flags make a long window *worse*, so do not improvise this set.
- **Screen every window** on the returned trace, on both the median and the log-spread. The seed
  state decides usability, and a median-only screen has passed a gradient wrong by five orders of
  magnitude.
- `boundary_stride ≈ √n_chunks` trades one forward pass for the host memory that otherwise makes a
  year impossible.
- Then check against finite differences anyway. Expect about 4 % at 60 days, and decide whether
  that is enough for what you are doing with it.

Related: `docs/ADJOINT_HORIZON.md` for the full measurement history, `docs/ADJOINT_API.md` for why
the interface is shaped this way, and notebook 4 for the flags on a small mesh you can iterate on
in seconds.